# Milestone 1 Lab, notebook 2 of 6: Your first LLM calls

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| **2. Your first LLM calls** (you are here) | Call a model, get a structured reply, and have AI draft the metric definitions |
| 3. Ask the LLM with all the data | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **2**. With the default model, the whole lab costs a few US cents.

## What this notebook is for

If you have never called a language model from code, this notebook is your starting point. We make two calls:

1. A plain call: text in, text out.
2. A structured call: text in, a filled-in form out.

We use them for a real data engineering job: turning the business's loose wording for "contribution profit" into a precise definition.

In [4]:
import os

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from lab_helpers import definitions, get_section, show_doc
from openai import OpenAI
from pydantic import BaseModel, Field

## Step 1: Connect

A large language model (LLM) runs on someone else's computers. We reach it over the internet: we send text, and it sends text back.

**OpenRouter** is a service that gives access to many models with one key. Your key is in the `.env` file in the project root. `load_dotenv` reads that file, and `OpenAI(...)` creates the object we use to send requests. (The library is called `openai` because OpenRouter speaks the same language as OpenAI's service.)

In [ ]:
load_dotenv(find_dotenv(usecwd=True), override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
MODEL = os.getenv("MODEL_NAME", "anthropic/claude-haiku-5.5")

if not api_key:
    raise RuntimeError("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file in the project root.")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key, timeout=120, max_retries=5)
print(f"Connected. Using model {MODEL}")

The model is chosen by `MODEL_NAME` in your `.env`. The lab is tested with `anthropic/claude-haiku-5.5`, a small and fast paid model. At its prices the whole lab costs a few US cents.

You can pay nothing by setting `MODEL_NAME=openrouter/free`, which picks a different free model for each request. Be warned that free models are slow, are limited to about 50 requests a day, and often fail at the structured calls in the later notebooks.

> **If you get an error** like "new accounts are limited to 20 requests per minute", wait a minute and run the cell again.

## Step 2: A plain call

We send a list of **messages**. Each message has a `role` and some `content`:

- the **system** message tells the model how to behave
- the **user** message is the request

The model's reply comes back in `response.choices[0].message.content`.

In [6]:
messages = [
    {"role": "system", "content": "You are a retail finance analyst. Answer in two sentences."},
    {"role": "user", "content": "What is contribution profit?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by recursive/flint-v1.0:free


Contribution profit is the revenue remaining after deducting variable costs, representing how much a product or sale contributes toward covering fixed costs and generating profit. In retail, it helps analysts evaluate whether sales are profitable by separating variable expenses from fixed overheads.

That is a whole LLM call: three lines.

The answer probably sounds right. But could you calculate contribution profit from it? Which costs exactly are taken off? If a customer is refunded in October for a September order, which month does the refund count in? The reply doesn't say, and a different run might say something different.

A plain reply is fine for reading. It is no good for building on. For that we need the reply in a fixed shape.

## Step 3: A structured call

With **structured output**, we give the model a form to fill in, and it replies with the form filled in.

We describe the form as a Python class (using a library called Pydantic). Each field has a name, a type, and a short description that the model reads.

Our form asks for two things: a draft formula for each metric, and the **ambiguities**, meaning the questions a person must answer before the definition can be trusted.

In [7]:
class DraftMetric(BaseModel):
    """One metric, as a first draft."""

    name: str = Field(description="snake_case name, such as net_sales")
    formula: str = Field(description="The formula, using other metric or component names")


class DraftDefinitions(BaseModel):
    """Draft metric definitions and the open questions behind them."""

    metrics: list[DraftMetric]
    ambiguities: list[str] = Field(description="Questions a person must answer before these definitions can be trusted")

The prompt gives the model the business's own wording, straight from the BRD.

In [ ]:
business_wording = get_section("BRD.md", "7. Profit definitions (business wording)").split("\n\nThe exact formulas")[0]

prompt = f"""You are helping a data engineer at a retailer turn business wording into precise metric definitions.

Here is the business wording:

{business_wording}

Draft a definition for net sales, gross profit, and contribution profit.
Then list the ambiguities: what the wording leaves open that would change the numbers.
"""

print(prompt)

The call is almost the same as before. Two things change: we use `parse` in place of `create`, and we pass our form as `response_format`. The filled-in form comes back in `.parsed`.

(The `extra_body` line tells OpenRouter to pick a model that supports structured output.)

In [9]:
response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    response_format=DraftDefinitions,
    extra_body={"provider": {"require_parameters": True}},
)
draft = response.choices[0].message.parsed

print(f"Drafted by {response.model}")
pd.DataFrame([metric.model_dump() for metric in draft.metrics])

Drafted by liquid/lfm-2.5-2.6b:free


,name,formula
0,Net Sales,Net Sales = Total Revenue from product sales i...
1,Gross Profit,Gross Profit = Net Sales - Cost of Goods Sold ...
2,Contribution Profit,Contribution Profit = Gross Profit - Variable ...


`draft` is a normal Python object now, not a block of text. We can loop over it, put it in a table, or save it. Here are the ambiguities the model raised.

In [10]:
for question in draft.ambiguities:
    print("-", question)

- Net Sales: The definition does not explicitly state whether 'discounts' include cash discounts (e.g., 2/10 net 30) or promotional markdowns. It also does not clarify if 'refunds' include store credit or partial refunds, which could affect the net amount.
- Gross Profit: The definition relies on the external cost of goods sold (COGS). It does not specify whether COGS includes freight-in (cost to get goods to the warehouse) or if it is strictly the purchase price of the units. The text implies COGS is the vendor payment for the units, but the inclusion of inbound logistics is ambiguous.
- Contribution Profit: The list of variable costs ('fulfillment, shipping, payment fees, and handling returns') is not exhaustive. It is unclear if there are other variable costs associated with each order (e.g., sales commissions, packaging materials, or customer service costs) that are not captured in this specific list.


> **If that cell failed**, run it again. Models occasionally fail to fill in a form, and free models do so often.

## Step 4: AI drafts, a person decides

Did the model ask when a refund should count? Did it ask what to do when a cost is missing? Those are exactly the kinds of question the business had to settle. Here are the decisions that were made.

In [11]:
show_doc("BRD.md", "6. Business rules agreed with the learner")

> **From `docs/BRD.md`: 6. Business rules agreed with the learner**

| Rule | Decision |
|---|---|
| Retailer and catalogue | Kestrel Ridge Outfitters, 12 products in 3 categories (Hydration, Camp, Apparel) |
| Time covered | Two months: September 2026 and October 2026 |
| "Last month" | October 2026. It is fixed by the dataset, not by today's date, so answers are repeatable |
| Currency | US dollars, one currency only |
| Refund timing | A refund reduces net sales in the month the refund is issued, not the month of the original order |
| Cancelled orders | An order cancelled before shipping is not a sale and is not counted |
| Missing information | If a figure cannot be calculated, it is reported as unavailable. It is never estimated or treated as zero |

And here are the exact formulas that came out of those decisions. This is our first look at the **LLD** (low-level design), the document with the precise detail.

In [12]:
show_doc("LLD.md", "2. Metric definitions (version 1.0.0)")

> **From `docs/LLD.md`: 2. Metric definitions (version 1.0.0)**

All amounts are US dollars, rounded to 2 decimal places. A month is written `YYYY-MM`.

| Term | Formula |
|---|---|
| `units_sold` | Units on order items placed in the month, excluding cancelled orders |
| `gross_sales` | Sum of quantity × list price for items ordered in the month, excluding cancelled orders |
| `discounts` | Sum of discounts on those items |
| `refunds` | Sum of refund amounts issued in the month |
| **`net_sales`** | `gross_sales − discounts − refunds` |
| `product_cost` | Sum of units sold × unit cost for those items |
| **`gross_profit`** | `net_sales − product_cost` |
| `fulfillment_cost` | Pick-and-pack cost for orders shipped in the month |
| `shipping_cost` | Carrier cost for orders shipped in the month |
| `payment_fees` | Payment provider fees on payments captured in the month |
| `return_cost` | Cost of handling returns for refunds issued in the month |
| **`contribution_profit`** | `gross_profit − fulfillment_cost − shipping_cost − payment_fees − return_cost` |

Rules:

- If any component of a metric is missing, the metric is missing (null). Nothing is estimated or treated as zero.
- Returned units are not resaleable, so `product_cost` is not reduced by returns.
- The three bold metrics are the approved metrics. The other nine terms are components and can be reported alongside them.

`metrics_v1.json` holds, for each of the twelve terms: `name`, `version`, `approved`, `unit`, `description`, `formula`, `components`, `assumptions`.

The agreed definitions are also stored as a file with a version number, so that code can read them. From here on, that file is the only source of truth for what a metric means.

In [13]:
print("Version", definitions["version"])
pd.DataFrame(definitions["metrics"])[["name", "approved", "formula"]]

Version 1.0.0


,name,approved,formula
0,units_sold,False,count of units on non-cancelled order items pl...
1,gross_sales,False,sum(quantity x list_price) for non-cancelled o...
2,discounts,False,sum(discount) on the order items counted in gr...
3,refunds,False,sum(refund_amount) for refunds issued in the m...
4,net_sales,True,gross_sales - discounts - refunds
5,product_cost,False,sum(units_sold x unit_cost)
6,gross_profit,True,net_sales - product_cost
7,fulfillment_cost,False,sum(pick_and_pack_cost) for orders shipped in ...
8,shipping_cost,False,sum(carrier_cost) for orders shipped in the month
9,payment_fees,False,sum(payment_fee) for payments captured in the ...


## What we now know

- **An LLM call** is a list of messages sent to a model. The system message sets behaviour, the user message asks.
- **Structured output** makes the model reply in a shape we define, so code can use the reply.
- **AI is a good drafting partner.** It wrote formulas in seconds and pointed out what the business left unsaid. But it did not decide anything. A person decided, and the decision was written down with a version number.

That last point is the pattern for this whole project: AI helps at every step, and anything that must be exact is fixed in code or in a file.

**Next: notebook 3**, where we ask the model the leader's actual question.